<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Hyperparameter Optimization for Keras with Scikit-Learn**


Neural networks contain many hyperparameters, such as:

- learning rate (extremely important)
- batch size (very important)
- number of neurons
- number of hidden layers
- optimizer type
- number of epochs

These hyperparameters can strongly affect model performance.

## **Scikit-Learn**
Instead of manually testing many combinations, we can use automated search methods such as:

- `GridSearchCV`
- `RandomizedSearchCV`

`GridSearchCV` tries every possible combination of hyperparameters. Grid search is less common for large deep learning models

For example:
{
    "batch_size": [16, 32],
    "epochs": [10, 20]
}

Grid Search tests:

```text
16 + 10
16 + 20
32 + 10
32 + 20
```
This is exhaustive but can become very slow when many hyperparameters are used.

`RandomizedSearchCV` tests only a random subset of combinations.

It is usually much faster than Grid Search and often finds good hyperparameters with much less computation.


## **Keras and Scikit-Learn Integration**
Keras models can be integrated with Scikit-Learn hyperparameter search tools by using a wrapper such as `KerasClassifier` or `KerasRegressor` from `scikeras.wrappers`. These wrappers allow a Keras neural network model to behave like a standard Scikit-Learn estimator.

This integration makes it possible to use Scikit-Learn tools such as:

- GridSearchCV
- RandomizedSearchCV
- cross-validation utilities

for neural network hyperparameter tuning.
After wrapping the Keras model, we can use familiar Scikit-Learn methods and attributes such as:
```python
fit()
best_score_
best_params_
```

## **Keras Tuners: Offer Its Own Grid Search**

TensorFlow/Keras provides its own hyperparameter tuning library called: `KerasTuner`

`KerasTuner` supports:
- Grid Search
- Random Search
- Bayesian Optimization
- Hyperband

KerasTuner is designed specifically for neural networks and is often more flexible than Scikit-Learn wrappers.

However, Scikit-Learn wrappers are still useful because:

- many users already know Scikit-Learn
- the syntax is simple and familiar
- it integrates easily with existing Scikit-Learn pipelines

## **Important Note About Runtime**

Hyperparameter optimization for neural networks can take a long time. This is because each parameter combination requires training an entire neural network.

For example:

```text
10 parameter combinations × 20 epochs each = 200 total training epochs
```

Large models or datasets may therefore require significant computation time.

## **Table of Contents**

1. [Objectives](#toc-objectives)
2. [Setup](#toc-setup)
    1. [Installing Required Libraries](#toc-installing-required-libraries)
    2. [Importing Required Libraries](#toc-importing-required-libraries)
    3. [Defining Helper Functions](#toc-defining-helper-functions)
3. [Create the Model](#toc-create-the-model)
    1. [Load the Data](#toc-load-the-data)
    2. [Data Wrangling](#toc-data-wrangling)
    3. [Build the Base Model](#toc-build-the-base-model)
4. [Randomized Search](#toc-randomized-search)
    1. [Parameters](#toc-parameters)
    2. [Define and Fit RandomizedSearchCV](#toc-define-and-fit-randomizedsearchcv)
    3. [Performance Evaluation](#toc-performance-evaluation)
5. [Exercises](#toc-exercises)
    1. [Exercise 1: Build the Base Model](#toc-exercise-1-build-the-base-model)
    2. [Exercise 2: Define Search Parameters](#toc-exercise-2-define-search-parameters)
    3. [Exercise 3: Fit RandomizedSearchCV](#toc-exercise-3-fit-randomizedsearchcv)


## **Objectives** <a id="toc-objectives"></a>

After completing this lab you will be able to:

*   Use Keras' scikit-learn wrapper to utilize sklearn functions on Keras models
*   Apply randomized search on Keras models to find the best hyperparameters


***


## **Setup** <a id="toc-setup"></a>


For this lab, we will be using the following libraries:

*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.
*   [`sklearn`](https://scikit-learn.org/stable/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and machine-learning-pipeline related functions.
*   [`matplotlib`](https://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for additional plotting tools.
*   [`tensorflow`](https://www.tensorflow.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and neural network related functions.


### **Installing Required Libraries** <a id="toc-installing-required-libraries"></a>

The following required libraries are pre-installed in the Skills Network Labs environment. However, if you run this notebook command in a different Jupyter environment (like Watson Studio or Anaconda), you will need to install these libraries by removing the `#` sign before `!mamba` in the following code cell.


In [1]:
# All Libraries required for this lab are listed below. The libraries pre-installed on Skills Network Labs are commented.
# !mamba install -qy numpy==1.21.4 matplotlib==3.5.0 scikit-learn==0.20.1
# Note: If your environment doesn't support "!mamba install", use "!pip install"

The following required libraries are **not** pre-installed in the Skills Network Labs environment. **You will need to run the following cell** to install them:


In [2]:
#!mamba install -qy tqdm

In [3]:
# Upgrade to newest version of skillsnetwork for more functionalities
# Restart kernel after doing so
# !pip install --upgrade skillsnetwork

### **Importing Required Libraries** <a id="toc-importing-required-libraries"></a>


In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # tensorflow INFO and WARNING messages are not printed
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from tqdm import tqdm
import numpy as np
%matplotlib inline

import tensorflow as tf
import keras
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from scikeras.wrappers import KerasClassifier

import skillsnetwork

### **Defining Helper Functions** <a id="toc-defining-helper-functions"></a>


In [5]:
# Vectorize integer sequence
def vectorize_sequence(sequence, dimensions):
    results = np.zeros((len(sequence), dimensions))
    for index, value in enumerate(sequence):
        if max(value) < dimensions:
            results[index, value] = 1
    return results


# Convert label into one-hot format
def one_hot_label(labels, dimensions):
    results = np.zeros((len(labels), dimensions))
    for index, value in enumerate(labels):
        if value < dimensions:
            results[index, value] = 1
    return results

## **Create the Model** <a id="toc-create-the-model"></a>


### **Load the Data** <a id="toc-load-the-data"></a>


For this exercise, we will be using the [Reuters newswire classification dataset](https://keras.io/api/datasets/reuters/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML311Coursera35714171-2022-01-01) from Keras. The training features for this dataset are lists of word indices (integers), corresponding to their frequency in the dataset. The response labels take on one of 46 classes, representing the newswire's topic.


In [6]:
await skillsnetwork.prepare(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module2/L1/reuters.npz",
    overwrite=True,
)

  0%|          | 0/2 [00:00<?, ?it/s]

Saved to '.'


In [7]:
X = np.load("x.npy", allow_pickle=True)
y = np.load("y.npy", allow_pickle=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

To get the word for a specific index, we can also extract a dictionary of words to index using the following Keras function.


In [8]:
word_to_ind = tf.keras.datasets.reuters.get_word_index(
    path="reuters_word_index.json"
)

### **Data Wrangling** <a id="toc-data-wrangling"></a>


Since each observation is a list of words that appear in the newswire, the length varies. Hence, we will vectorize the dataset using `vectorize_sequence()` to ensure that all inputs to our model have the same dimension. Labels are also one-hot encoded with `one_hot_label()` because classes (news topic) are not ordinal.


In [9]:
dim_x = max([max(sequence) for sequence in X_train]) + 1
dim_y = max(y_train) + 1

X_train_vec = vectorize_sequence(X_train, dim_x)  # Vectorize the training data
X_test_vec = vectorize_sequence(X_test, dim_x)
y_train_hot = one_hot_label(
    y_train, dim_y
)  # Convert the training labels into one-hot format
y_test_hot = one_hot_label(y_test, dim_y)

### **Build the Base Model** <a id="toc-build-the-base-model"></a>


In order to apply `RandomizedSearchCV` on Keras models, we will be using `KerasClassifier` from `scikeras.wrappers`, which will let us apply scikit-learn functions on the model.


We define `create_model()` below to detail which layers we want to include in the model. Recall that the final Dense layer has 46 units to correspond to the number of classes. This also prompts us to use categorical cross entropy as a loss function. Here, `neuron` is included as a parameter with default value because we want to tune it later.


In [10]:
# Create Keras Sequential Model as base model
def create_model(neurons=10):
    model = Sequential()
    model.add(Input(shape=(dim_x,)))
    model.add(Dense(neurons, activation="linear"))
    model.add(Dense(64, activation="relu"))
    model.add(Dense(46, activation="softmax"))
    model.compile(
        optimizer="RMSprop",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

For the base model, we won't change any parameters so that we can compare them with results after hyperparameter tuning. We also specify some of the default values for hyperparameters that don't appear in `create_model()` (example batch_size, epochs) such that they are defined when applying randomized search.


In [11]:
np.random.seed(0)
base_model = KerasClassifier(
    model=create_model, verbose=0, batch_size=10, epochs=1
)

Fitting the model on the train set, we obtain the test score for our base model.


In [12]:
# Get pre-tuned results
base_model.fit(X_train_vec, y_train_hot)
base_score = base_model.score(X_test_vec, y_test_hot)
print("The baseline accuracy is: %.3f" % base_score)

The baseline accuracy is: 0.743


## **Randomized Search** <a id="toc-randomized-search"></a>


### **Parameters** <a id="toc-parameters"></a>


As you might already know from performing randomized search on machine learning models, we have to create a dictionary for the hyperparameter values. Let's start by defining the values we want to experiment with! Note that if you would like to test other parameters, they must be defined in the base model as well.


In [13]:
batch_size = [10, 20, 60, 80]
epochs = [1, 3, 5]
neurons = [1, 10, 20, 30]

params = dict(batch_size=batch_size, epochs=epochs, model__neurons=neurons)
params

{'batch_size': [10, 20, 60, 80],
 'epochs': [1, 3, 5],
 'model__neurons': [1, 10, 20, 30]}

### **Define and Fit RandomizedSearchCV** <a id="toc-define-and-fit-randomizedsearchcv"></a>


In [14]:
search = RandomizedSearchCV(
    estimator=base_model, param_distributions=params, cv=3
)

Now, fit randomized search on `X_train_vec` and `y_train_hot` as you would for any other model. **Note that this may take a while to run (10+ minutes)**, especially if there are a lot of parameter combinations, or if the epoch size is big. If you have the resources, you could also switch out `RandomizedSearchCV` for `GridSearchCV` to search over every combination of hyperparameters (takes even more time to run).


In [15]:
search_result = search.fit(X_train_vec, y_train_hot)

### **Performance Evaluation** <a id="toc-performance-evaluation"></a>


Let's take a look at the results from this search! In particular, we will examine the mean and standard deviation of the cross-validation score under different hyperparameter combinations.


In [16]:
means = search_result.cv_results_["mean_test_score"]
stds = search_result.cv_results_["std_test_score"]
params = search_result.cv_results_["params"]

`RandomizedSearchCV` also has attributes for us to access the best score and parameters directly.


In [17]:
print(
    "Best mean cross-validated score: {} using {}".format(
        round(search_result.best_score_, 3), search_result.best_params_
    )
)

Best mean cross-validated score: 0.791 using {'model__neurons': 30, 'epochs': 3, 'batch_size': 10}


We can also print out all the other scores:


In [18]:
for mean, stdev, param in zip(means, stds, params):
    print(
        "Mean cross-validated score: {} ({}) using: {}".format(
            round(mean, 3), round(stdev, 3), param
        )
    )

Mean cross-validated score: 0.724 (0.017) using: {'model__neurons': 10, 'epochs': 3, 'batch_size': 60}
Mean cross-validated score: 0.558 (0.008) using: {'model__neurons': 1, 'epochs': 3, 'batch_size': 10}
Mean cross-validated score: 0.679 (0.005) using: {'model__neurons': 20, 'epochs': 1, 'batch_size': 60}
Mean cross-validated score: 0.769 (0.003) using: {'model__neurons': 20, 'epochs': 3, 'batch_size': 60}
Mean cross-validated score: 0.481 (0.073) using: {'model__neurons': 1, 'epochs': 5, 'batch_size': 60}
Mean cross-validated score: 0.606 (0.015) using: {'model__neurons': 10, 'epochs': 1, 'batch_size': 80}
Mean cross-validated score: 0.786 (0.007) using: {'model__neurons': 20, 'epochs': 5, 'batch_size': 60}
Mean cross-validated score: 0.364 (0.012) using: {'model__neurons': 1, 'epochs': 3, 'batch_size': 80}
Mean cross-validated score: 0.791 (0.001) using: {'model__neurons': 30, 'epochs': 3, 'batch_size': 10}
Mean cross-validated score: 0.788 (0.005) using: {'model__neurons': 20, 'epo

From this, we can see how different the other models' scores are compared to the optimal model's performance. Some are pretty close to the best score, whereas there are combinations that yield much lower scores.Thank goodness we didn't pick those! With randomized search on neural networks, we are able to determine the best values in an automated way.


Using the best estimator, let's get the test score:


In [19]:
print(
    "Best test score: %.3f"
    % search_result.best_estimator_.score(X_test_vec, y_test_hot)
)

Best test score: 0.816


Our test score has increased compared to the base model!


## **Exercises** <a id="toc-exercises"></a>


Now, let's try Randomized search on other hyperparameters!


### **Exercise 1: Build the Base Model** <a id="toc-exercise-1-build-the-base-model"></a>


This time, we want to look at different optimizers, learning rates, and dropout rates. Since the learning rate is a value fed into `optimizer`, the parameter has to be named a certain way in `create_model()` and `params`: `model__optimizer__learning_rate`.


In [20]:
# Create Keras Sequential Model as base model
def create_model(
    optimizer="RMSprop", optimizer__learning_rate=0.1, dropout_rate=0.2
):
    optimizer = tf.keras.optimizers.get(optimizer)
    optimizer.learning_rate = optimizer__learning_rate
    model = Sequential()
    model.add(Input(shape=(dim_x,)))
    model.add(Dense(64, activation="linear"))
    model.add(Dropout(dropout_rate))
    model.add(Dense(64, activation="relu"))
    model.add(Dense(46, activation="softmax"))
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [21]:
# TODO: Create `base_model` using `KerasClassifier`. Fit it to the training set and obtain the test score
np.random.seed(0)
base_model = KerasClassifier(
    model=create_model, verbose=0, batch_size=10, epochs=1
)

In [22]:
# Get pre-tuned results
base_model.fit(X_train_vec, y_train_hot)
base_score = base_model.score(X_test_vec, y_test_hot)
print("The baseline accuracy is: %.3f" % base_score)

The baseline accuracy is: 0.344


### **Exercise 2: Define Search Parameters** <a id="toc-exercise-2-define-search-parameters"></a>


Now, we will specify which values we want to experiment with and put them into a dictionary.


In [23]:
# TODO: specify 3-4 values for each of the following parameters
optimizer = ["SGD", "RMSprop", "Adam"]
learning_rate = [0.01, 0.1, 1]
dropout_rate = [0.1, 0.3, 0.6, 0.9]

# TODO: convert it into a dictionary variable named `params`
params = dict(
    model__optimizer=optimizer,
    model__optimizer__learning_rate=learning_rate,
    model__dropout_rate=dropout_rate,
)

### **Exercise 3: Fit RandomizedSearchCV** <a id="toc-exercise-3-fit-randomizedsearchcv"></a>
**(Note that this may take a while to run (5+ minutes))**


In [24]:
# TODO: Create `RandomizedSearchCV` object `search` and fit it to the train set
search = RandomizedSearchCV(
    estimator=base_model, param_distributions=params, cv=3
)

search_result = search.fit(X_train_vec, y_train_hot)


In [25]:
# TODO: Obtain the best cross-validation score and test scores
for mean, stdev, param in zip(means, stds, params):
    print(
        "Mean cross-validated score: {} ({}) using: {}".format(
            round(mean, 3), round(stdev, 3), param
        )
    )

Mean cross-validated score: 0.724 (0.017) using: model__optimizer
Mean cross-validated score: 0.558 (0.008) using: model__optimizer__learning_rate
Mean cross-validated score: 0.679 (0.005) using: model__dropout_rate


## **Authors**


[Cindy Huang](https://www.linkedin.com/in/cindy-shih-ting-huang/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML311Coursera35714171-2022-01-01) is a data science associate of the Skills Network team. She has a passion for machine learning to improve user experience, especially in the area of computational linguistics.


### **Other Contributors**


Su Wu

## **Change Log**


| Date (YYYY-MM-DD) | Version | Changed By | Change Description |
| ----------------- | ------- | ---------- | ------------------ |
| 2022-07-20        | 1.0     | Cindy H.   | Create lab draft   |
| 2022-09-01        | 1.0     | Steve Hord | QA pass edits      |
| 2022-11-11        | 1.0     | Shengkai C.| Review and edit    |
| 2026-05-19        | 1.0     | Su W.| Documentation edit    |

Copyright © 2022 IBM Corporation. All rights reserved.
